# Calculate the difference between the Taunton-Worcester scenario and base

Gather relevant columns.

## Code setup

In [1]:
import os
import numpy as np
import pandas as pd

In [2]:
dir_data = 'data'
dir_output = 'output'
# Scenario results:
file_base_geo = 'results_geo_outcomes_base.csv'
file_base_results = 'results_full_base.csv'
file_taunworc_geo = 'results_geo_outcomes_taunworc.csv'
file_taunworc_results = 'results_full_taunworc.csv'
# Output:
file_results_comparison = 'results_full_comparison_taunworc_base.csv'

## Load data

### Scenario results

In [3]:
p = os.path.join(dir_output, file_base_geo)
p2 = os.path.join(dir_output, file_base_results)
df_base = pd.concat(
    (pd.read_csv(p, index_col='LSOA'), pd.read_csv(p2, index_col='LSOA')),
    axis='columns'
)

In [4]:
p = os.path.join(dir_output, file_taunworc_geo)
p2 = os.path.join(dir_output, file_taunworc_results)
df_taunworc = pd.concat(
    (pd.read_csv(p, index_col='LSOA'), pd.read_csv(p2, index_col='LSOA')),
    axis='columns'
)

In [5]:
display(df_base.head(5))

,nearest_ivt_unit,nearest_ivt_time,nearest_ivt_unit_no_bias,nearest_ivt_time_no_bias,nearest_mt_unit,nearest_mt_time,transfer_unit,transfer_required,transfer_time,area_lower,...,drip_ship_mrs_0-2_improvement_over_no_treatment,mothership_mrs_0-2_improvement_over_no_treatment,nlvo_drip_ship_ivt_mrs_0-2_improvement_over_no_treatment,lvo_drip_ship_ivt_mrs_0-2_improvement_over_no_treatment,lvo_drip_ship_ivt_mt_mrs_0-2_improvement_over_no_treatment,drip_ship_utility_shift_transferloss0!6,drip_ship_mrs_0-2_transferloss0!6,drip_ship_mrs_0-2_improvement_over_no_treatment_transferloss0!6,lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!6,lvo_drip_ship_ivt_mt_mrs_0-2_improvement_over_no_treatment_transferloss0!6
LSOA,,,,,,,,,,,,,,,,,,,,,
ADUR001A,BN25BE,10.9,BN25BE,11,BN25BE,11,BN25BE,False,0,Adur 001A,...,0.134,0.133,0.121,0.06,0.164,0.12470,0.619,0.134,0.429,0.164
ADUR001B,BN25BE,11.9,BN25BE,12,BN25BE,12,BN25BE,False,0,Adur 001B,...,0.133,0.133,0.120,0.06,0.164,0.12424,0.619,0.133,0.429,0.164
ADUR001C,BN112DH,11.0,BN112DH,11,BN25BE,13,BN25BE,True,24,Adur 001C,...,0.117,0.133,0.120,0.06,0.111,0.10364,0.597,0.111,0.356,0.091
ADUR001D,BN25BE,11.9,BN25BE,12,BN25BE,12,BN25BE,False,0,Adur 001D,...,0.133,0.133,0.120,0.06,0.164,0.12424,0.619,0.133,0.429,0.164
ADUR001E,BN112DH,10.0,BN112DH,10,BN25BE,13,BN25BE,True,24,Adur 001E,...,0.118,0.133,0.121,0.06,0.112,0.10407,0.598,0.112,0.356,0.091


## Gather data

In [6]:
cols_same = ['area_lower', 'Admissions_2025_Est']

cols_to_gather = ['nearest_ivt_unit', 'transfer_unit', 'transfer_required',
                  'drip_ship_mt_travel_time',
                  'drip_ship_mrs_0-2', 'lvo_drip_ship_ivt_mt_mrs_0-2',
                  'drip_ship_mrs_0-2_transferloss0!6',
                  'lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!6']

rename_dict_base = dict([(c, f'{c}_base') for c in cols_to_gather])
rename_dict_taunworc = dict([(c, f'{c}_taunworc') for c in cols_to_gather])

df = pd.concat(
    (df_base[cols_same],
     df_base[cols_to_gather].rename(columns=rename_dict_base),
     df_taunworc[cols_to_gather].rename(columns=rename_dict_taunworc)),
    axis='columns'
)

Calculate changes from base:

In [9]:
cols_for_change = ['drip_ship_mt_travel_time', 'drip_ship_mrs_0-2',
                   'lvo_drip_ship_ivt_mt_mrs_0-2',
                   'drip_ship_mrs_0-2_transferloss0!6',
                   'lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!6']

for col in cols_for_change:
    df[f'change_in_{col}'] = df[f'{col}_taunworc'] - df[f'{col}_base']

df['changed'] = (
    (df['nearest_ivt_unit_base'] != df['nearest_ivt_unit_taunworc']) |
    (df['transfer_unit_base'] != df['transfer_unit_taunworc'])
)
# df['change_in_lvo_drip_ship_ivt_mt_mrs_0-2'] != 0

Save to file:

In [10]:
p = os.path.join(dir_output, file_results_comparison)
df.to_csv(p)